In [1]:
# import the required packages
import pandas as pd
import numpy as np

In [2]:
# Load the SQL toolset extension
%load_ext sql

In [3]:
#2. Connect to (and create) the database file
%sql sqlite:///bike_orders_database.sqlite

Connecting to 'sqlite:///bike_orders_database.sqlite'

In [4]:
def collect_data():
    """
    Collects and combines the bike orders data using the jupyter-sql magic extension.
    """
    
    # 1.0 Connect to database and load tables via SQL magic
    # Using the %sql magic command to fetch data directly into DataFrames
    bikes     = %sql SELECT * FROM bikes
    bikeshops = %sql SELECT * FROM bikeshops
    orderlines = %sql SELECT * FROM orderlines
    
    # Convert to DataFrames and drop 'index' column
    data_dict = {
        'bikes': bikes.DataFrame().drop("index", axis=1),
        'bikeshops': bikeshops.DataFrame().drop("index", axis=1),
        'orderlines': orderlines.DataFrame().drop("index", axis=1)
    }

    # 2.0 Combining Data
    joined_df = data_dict['orderlines'] \
        .merge(
            right    = data_dict['bikes'],
            how      = 'left',
            left_on  = 'product.id',
            right_on = 'bike.id'
        ) \
        .merge(
            right    = data_dict['bikeshops'],
            how      = "left",
            left_on  = "customer.id",
            right_on = 'bikeshop.id'
        )

    # 3.0 Cleaning Data 
    df = joined_df
    df['order.date'] = pd.to_datetime(df['order.date'])

    temp_df = df['description'].str.split(" - ", expand = True)
    df['category.1'] = temp_df[0]
    df['category.2'] = temp_df[1]
    df['frame.material'] = temp_df[2]

    temp_df = df['location'].str.split(", ", expand = True)
    df['city'] = temp_df[0]
    df['state'] = temp_df[1]

    df['total.price'] = df['quantity'] * df['price']

    cols_to_keep_list = [
        'order.id', 'order.line', 'order.date',    
        'quantity', 'price', 'total.price', 
        'model', 'category.1', 'category.2', 'frame.material', 
        'bikeshop.name', 'city', 'state'
    ]

    df = df[cols_to_keep_list]
    df.columns = df.columns.str.replace(".", "_")

    return df

In [5]:
df = collect_data()

Running query in 'sqlite:///bike_orders_database.sqlite'

Running query in 'sqlite:///bike_orders_database.sqlite'

Running query in 'sqlite:///bike_orders_database.sqlite'

# 1. Selecting Columns

### Select by name
![Select by name](subset_cols.png)

In [ ]:
df

In [ ]:
[1,2,3, "hello", [7,8,9]]
df['order_date']

In [ ]:
# Select by name
# Subsetting data frame columns with a list of names
['order_date', 'order_id', 'order_line'] # make a list
df[['order_date', 'order_id', 'order_line']] # get the specific cols in the order we pass them
# df['order_date', 'order_id'] # we will get an error. Pandas is trying to index those columns
df['order_date'] # we will get a series with a single bracket
type(df['order_date'])
df[['order_date']] # we will get a data frame with double brackets
type(df[['order_date']])

### Select by position
In pandas, *loc* and *iloc* are the primary tools for selecting data from a DataFrame. The fundamental difference lies in how they identify the data: *loc* is label-based, while *iloc* is integer-position-based.
```python
df.loc[row_label, column_label]
df.iloc[row_position, column_position]
```
![Select by position](subset_iloc.png)

In [ ]:
df

In [ ]:
# Select by position
# iloc
df.iloc[:, 0:3] # select all rows and cols 0 to 3
df.iloc[:, -3:] # Last 3

# loc
# Select all rows where price is greater than 12000
df.loc[df['price'] > 12000]

In [ ]:
df.iloc[:100, -3:]

In [ ]:
# Select by text matching
# Use df.filter().  Filter is based on column or row index using text patterns
# The pandas filter() function is a specialized tool used to subset a DataFrame or Series based on labels 
# rather than values. It is highly useful when you need to select columns or index rows that match 
# specific naming patterns, such as prefixes or suffixes.
# If they try to use it to filter rows based on the content of a column (like filter(quantity > 5)), 
# it will throw an error. For content-based filtering, they should use boolean indexing instead 
# (e.g., df[df['quantity'] > 5]).
# Most of the time I use:
#  "term" = Contains "term"
#  "^term" = Starts with "term"
#  "term$" = Ends with "term"
#  (term1)|(term2) = Matches "term1" or "term2"
df.filter(regex="(^model)|(^cat)", axis=1)  # everything that starts with model or starts with cat
df.filter(regex="(id$)", axis=1)
df.filter(regex="(price)", axis=1)

In [ ]:
?pd.DataFrame.filter

In [ ]:
# Rearranging a single column
# Move 'model' to the front
df.columns
df.columns.tolist()  # converts a column or row index to a list, which is iterable
l = df.columns.tolist()
l.remove('model')
l  # l does not have the model column now
# df[['model', l]] # I want the column model first.  This is going to throw an error
['model', l] # I get a nested list
['model', *l] # I unpack the list
df[['model', *l]] # unpack the list

In [ ]:
# Rearranging multiple columns
# Move 'model', 'category_1', and 'category_2' to the front
l = df.columns.tolist()
l
l.remove('model')  # removes only removes one column at a time
l.remove('category_1')
l.remove('category_2')
l
['model', 'category_1', 'category_2', *l]
df[['model', 'category_1', 'category_2', *l]]

In [ ]:
# Rearranging multiple columns (list comprehension)
l = df.columns.tolist()
l
['model', 'category_1', 'category_2']
cols_to_front = ['model', 'category_1', 'category_2']
# Step by step list comprehension
[col for col in l]  # col is temporary variable that gets assigned during the loop. l is the list we are looping through
[col for col in l if col not in cols_to_front]  # it removes the cols in cols_to_front from l
l2 = [col for col in l if col not in cols_to_front]  # assign it to l2
[cols_to_front, l2] # we get 2 nested lists
[*cols_to_front, *l2] # unpack both lists
df[[*cols_to_front, *l2]]

In [ ]:
# Select by data types
# df.select_dtypes(): Select columns and subset by dtype (data type)
df.info()

df.select_dtypes(include='str')
df1 = df.select_dtypes(include='str')

df.select_dtypes(exclude='str')
df2 = df.select_dtypes(exclude='str')

# Concatenate data frames
# pd.concat(): Combines multiple data frames contained in a list
pd.concat([df1, df2], axis=1) # it accepts a list.  Provide the axis, 1 is for cols

# Drop columns
# df.drop(): Exclude columns by name
df[['model', 'category_1', 'category_2']]
df.drop(['model', 'category_1', 'category_2'], axis=1)

# move ['model', 'category_1', 'category_2'] to the front
df3 = df[['model', 'category_1', 'category_2']]
df4 = df.drop(['model', 'category_1', 'category_2'], axis=1)
pd.concat([df3, df4], axis=1)

In [ ]:
?df.select_dtypes

# 2. Arranging Rows

In [ ]:
# Sorting
# df.sort_values(): Great for sorting by a column. Can sort by multiple columns to get group-wise sorts.
df.sort_values('total_price')
df.sort_values('total_price', ascending=False)
df.sort_values('order_date') # it is already sorted
df.sort_values('order_date', ascending=False)

# Sorting a Series
df['price'].sort_values(ascending=False)  # No need to pass the column name since this is a Series

# 3. Rowwise Filtering (Slicing)

In [ ]:
# Simple filters with booleans
# The key concept is that we need to create a "Boolean Series", a pandas Series that consists only of
# True/False values. We then use this to subset our data frame rowwise
df.order_date
df.order_date >= pd.to_datetime("2015-01-01")  # We first create the boolean series
df[df.order_date >= pd.to_datetime("2015-01-01")]

df.model == "Trigger Carbon 1"
df[df.model == "Trigger Carbon 1"]

# Use string accessors
# s.str.startswith(): We can use the string accessor of pandas series containing string data.
# The string accessor has startswith(), contains(), and endswith() methods.
df.model.str.startswith("Trigger")
df[df.model.str.startswith("Trigger")]

df.model.str.contains("Carbon")
df[df.model.str.contains("Carbon")]

In [ ]:
# Query filters
# df.query(): A string-based rowwise filtering method. It allows Boolean expressions for filtering.
# Works well in method chaining.
#  df.query('Length > 7')
# df.query('length > 7 and Width < 8')
# df.query('Name.str.startswith("abc")', engine="python")
price_threshold = 5000
df.query("price >= @price_threshold")

price_threshold_1 = 9000
price_threshold_2 = 1000
df.query("(price >= @price_threshold_1) | (price <= @price_threshold_2)")

f"price >= {price_threshold_1}"
df.query(f"price >= {price_threshold_1}") # cleaner to understand

In [ ]:
price_threshold = 5000

In [ ]:
# Filtering items in a list. Filtering with isin() and ~
# series.isin(): Detect whether values are in a list of values
df['category_2']
df['category_2'].unique()  # returns unique values of a Series object.
df['category_2'].value_counts()  # A frequency count

df['category_2'].isin(['Trail', 'Cyclocross'])  # Pass a list. We get a Boolean Series
df[df['category_2'].isin(['Trail', 'Cyclocross'])] 

# An alternative to using series.isin() is to use regex pattern with:
# series.str.contains(regex="(Trail) | (Cyclocross)")

# Negates using tilde (~).  It flips the boolean series to form the opposite True/False.
df[~df['category_2'].isin(['Trail', 'Cyclocross'])] 


In [ ]:
# Slicing
df[:5]  # Grab the first five rows. The equivalent of df.head(5)
df.tail(5)  # Grab the last five

# Index slicing with df.iloc[]
df.iloc[0:5] # Grab the first five rows
df.iloc[0:5, [1,3,5]]
df.iloc[0:5, :]  # Get all columns
df.iloc[:, [1,3,5]] # Get all rows and cols 1, 3, and 5

In [ ]:
# Unique / Distinct Values
# Drop Duplicates
# df.drop_duplicates(): remove duplicate rows returning the unique combinations
df[['model', 'category_1', 'category_2', 'frame_material']] \
    .drop_duplicates()  # We get the unique combinations.  Note that the df index has been reset

df['model'].unique()

In [ ]:
# Top / Bottom
# nlargest(): Finds the N-largest rows using a numeric column
df.nlargest(n=20, columns='total_price')
df['total_price']  # A series
df['total_price'].nlargest(n=20)  # We get the same indexes back than before

df.nsmallest(n=20, columns='total_price')

In [ ]:
# Sampling Rows
# df.sample(): Select random rows. Good for train/test splitting.  Use random_state to make reproducible.
df.sample(n=10)
df.sample(n=10, random_state=42)
df.sample(frac=0.10, random_state=42)

# 4. Adding Calculated Columns (Mutating)

In [ ]:
# Method 1: Series Notations
# The downside is that Series notation does not support Method Chaining
# df.copy(): Makes a copy of the data frame so we don't accidentally overwrite it
df2 = df.copy()
df2['new_col'] = df2['price'] * df2['quantity'] # we have this col in df named as total_price
df2['new_col2'] = df['model'].str.lower()

In [ ]:
# Method 2: Assign (Great for Method Chaining)
# df.assign(). Assign new columns to a DataFrame. It supports method chaining.  It just needs to follow the lambda function syntax
# Lambda Function: A funtion made on the fly that has no name. It is sometimes
# called an "Anonymous Function".
# Why do we use Lambda Functions? Less code, easier to read, don't need to think about what to name it.
# Example:

# def price_quantity(x):
#     return x.price * x.quantity

# Can be converted to a short-hand lambda function:
#  lambda x: x.price * x.quantity
df['frame_material']
df['frame_material'].str.lower()

# Important: Understand that "x" is always the incoming data frame.  We will access a Series using x['column'] notation
# The function matters: The code after the colon is what actually performs the work.
# Consistency: The only reason x is used so often is that it is short, fast to type, and clearly signals to other developers that this is a temporary, local, anonymous function.
# If a method documentation says "accepts a callable," it means you can pass a def function, a lambda, or a built-in function like sum. 
# It is one of the most powerful features for writing clean, professional data pipelines.
df.assign(frame_material = lambda x: x['frame_material'].str.lower()) # we are overwriting frame_material
df.assign(frame_material_lower = lambda x: x['frame_material'].str.lower()) # we are creating a new col named frame_material_lower

In [ ]:
# Method 2: Assign (continuation)
df[['model', 'price']]

df[['model', 'price']] \
    .drop_duplicates()

df[['model', 'price']] \
    .drop_duplicates() \
    .set_index('model')

df[['model', 'price']] \
    .drop_duplicates() \
    .set_index('model') \
    .plot(kind='hist')

# The plot is skewed.  When outliers are present, they skew data.The outliers can have a drastic effect 
# on linear regression models. One way to compensate for skew is to take a Log Transformation.
# We can implement it with df.assign()
df[['model', 'price']] \
    .drop_duplicates() \
    .assign(price = lambda x: np.log(x['price'])) \
    .set_index('model') \
    .plot(kind='hist')

In [6]:
# Adding Flags (True/False)
"Supersix Evo Hi-Mod Team"

# .find("supersix"): This is the string method that searches for the position of the substring "supersix".
# How .find() works: It returns the index (position) where the substring first begins.
# Since "supersix" starts at the very beginning of the new lowercase string, its index is 0.
"Supersix Evo Hi-Mod Team".lower().find("supersix")
"Supersix Evo Hi-Mod Team".lower().find("supersix") >= 0

"Beast of the East 1"
"Beast of the East 1".lower().find("supersix") # it returns -1, it means that it does not exist
"Beast of the East 1".lower().find("supersix") >= 0 # Returns False

# Pandas Series has the str.contains() method, which simplifies text searching, so we don't need to use find() >= 0
df.assign(flag_supersix = lambda x: x['model']) # It creates the col which is a copy of model
df.assign(flag_supersix = lambda x: x['model'].str.lower()) # same as before but lowercase
df.assign(flag_supersix = lambda x: x['model'].str.lower().str.contains("supersix"))

0

In [ ]:
# Binning
# A great way to convert numeric data to groups based on their values
# There are two main types:
# 1. Even-Width Binning, pd.cut()
# 2. Quantile Binning, pd.qcut()

pd.cut(df.price, bins = 3)
# Categorical data: Combines a text label with a numeric value (order)
pd.cut(df.price, bins = 3, labels = ['low', 'medium', 'high']) # Categories object
pd.cut(df.price, bins = 3, labels = ['low', 'medium', 'high']).astype("str") # String object

df[['model', 'price']]
df[['model', 'price']] \
    .drop_duplicates() 

df[['model', 'price']] \
    .drop_duplicates() \
    .assign(price_group = lambda x: pd.cut(x.price, bins=3))

In [ ]:
# Visualizing Binning Strategies with a Pandas Heat Table
# Pandas DF Style Accessor. Data frames have a style accessor that many data scientists don't know about.
# It is really powerful for making tables for HTML and Excel reports

# df.pivot(). Used to reshape a data frame from long to wide format
df[['model', 'price']] \
    .drop_duplicates() \
    .assign(price_group = lambda x: pd.cut(x.price, bins=3)) \
    .pivot(
        index = 'model',
        columns = 'price_group',
        values = 'price'
    )

df[['model', 'price']] \
    .drop_duplicates() \
    .assign(price_group = lambda x: pd.cut(x.price, bins=3)) \
    .pivot(
        index = 'model',
        columns = 'price_group',
        values = 'price'
    ) \
    .style.background_gradient(cmap = 'Blues')


In [ ]:
# Quantile Binning with pd.qcut()
# It adjusts the bin widths to evenly distribute observations across all bins.
# This method is used most of the time when binning

pd.qcut(df.price, q=[0, 0.33, 0.66, 1])
pd.qcut(df.price, q=[0, 0.33, 0.66, 1], labels=['low', 'medium', 'high'])

# Provide the quantiles using q
df[['model', 'price']] \
    .drop_duplicates() \
    .assign(price_group = lambda x: pd.qcut(x.price, q=3)) \
    .pivot(
        index = 'model',
        columns = 'price_group',
        values = 'price'
    ) \
    .style.background_gradient(cmap = 'Blues')

In [ ]:
# Close the connection
%sql --close sqlite:///bike_orders_database.sqlite